
# Exploratory data analysis of directed hippocampal E/I chains

This notebook analyzes **directed 3-node / 2-edge chains** in a signed, weighted hippocampal connectivity netlist such as `mij_netlist.csv`.

Primary outputs:

- Counts and percentages of all 8 E/I chain signatures: `EEE`, `EEI`, `EIE`, `EII`, `IEE`, `IEI`, `IIE`, `III`
- Absolute chain-weight totals and percentages
- Signed chain-weight totals
- Weighted-motif intensity (geometric mean)
- Region composition, overall and within each E/I signature
- Region pathway composition (for example `DG → CA3 → CA1`)
- Cell-type composition **after removing region prefixes**
- Cell-type pathway composition
- Position-specific breakdowns for source, middle, and target nodes
- Figures for every major summary
- CSV exports of all summary tables

## Definition of chain weight

For a chain \(A \rightarrow B \rightarrow C\) with edge weights \(w_1\) and \(w_2\):

\[
W_{\mathrm{signed}} = w_1w_2
\]

\[
W_{\mathrm{abs}} = |w_1w_2|
\]

\[
I = \sqrt{|w_1w_2|}
\]

`abs_chain_weight` is used for percentage-of-chain-weight calculations because signed sums can cancel excitation and inhibition. `signed_chain_weight` is retained separately.

The geometric-mean quantity follows the weighted-subgraph **intensity** concept introduced by Onnela et al. (2005). Because every chain here has the same number of edges, the raw product and geometric mean provide equivalent rankings, although their numerical scales differ.

A global positive rescaling of the entire matrix changes raw chain products but **does not change the percentage share of chain weight**, because every two-edge chain is multiplied by the same scale factor squared. Thus, global normalization is not required merely to calculate within-matrix chain-weight percentages.

### References

1. Onnela J-P, Saramäki J, Kertész J, Kaski K. *Intensity and coherence of motifs in weighted complex networks.* Physical Review E. 2005;71:065103. https://doi.org/10.1103/PhysRevE.71.065103
2. NetworkX documentation, `all_simple_paths`: a simple path contains no repeated nodes. https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.simple_paths.all_simple_paths.html

> This notebook uses a vectorized edge-join instead of `networkx.all_simple_paths` because only fixed-length, two-edge chains are required. The resulting definition is the same simple-path concept when `SIMPLE_CHAINS_ONLY=True`.


In [ ]:

from pathlib import Path
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)


## 1. Configuration

In [ ]:

# Input file. You may override this without editing the notebook:
#   Windows PowerShell: $env:MIJ_NETLIST_PATH="C:\path\mij_netlist.csv"
#   Bash:              export MIJ_NETLIST_PATH=/path/mij_netlist.csv
DATA_PATH = Path(os.getenv("MIJ_NETLIST_PATH", "mij_netlist.csv"))

# Expected columns in the current Hippocampome netlist.
COL = {
    "pre": "pre_neuron",
    "post": "post_neuron",
    "pre_ei": "pre_ei",
    "post_ei": "post_ei",
    "weight": "m_ij",
}

# If the netlist already contains explicit metadata columns, set these names.
# Leave as None to infer metadata from neuron labels.
OPTIONAL_COL = {
    "pre_region": None,       # e.g. "pre_region"
    "post_region": None,      # e.g. "post_region"
    "pre_layer": None,        # e.g. "pre_layer"
    "post_layer": None,       # e.g. "post_layer"
    "pre_cell_type": None,    # e.g. "pre_cell_type"
    "post_cell_type": None,   # e.g. "post_cell_type"
}

# A "simple" 3-node chain requires A, B, and C to be distinct.
# A->B->A return paths are excluded when True.
SIMPLE_CHAINS_ONLY = True

# Self-connections never constitute one of the two chain edges.
EXCLUDE_SELF_EDGES = True

# Duplicate directed edges between the same pair are summed before chain enumeration.
AGGREGATE_DUPLICATE_EDGES = True

# Plot/export controls.
TOP_N_CELL_TYPES = 20
TOP_N_PATHWAYS = 20
PLOT_EACH_SIGNATURE = True
SAVE_FIGURES = True
EXPORT_CHAIN_TABLE = False  # turn on only if you want the potentially large chain-level CSV

OUTPUT_DIR = Path("chain_eda_outputs")
FIG_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)

SIGNATURE_ORDER = ["EEE", "EEI", "EIE", "EII", "IEE", "IEI", "IIE", "III"]

print("DATA_PATH:", DATA_PATH.resolve())
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())



## 2. Helper functions

Region inference defaults to the **first whitespace-delimited token** in the Hippocampome label.

Layer tokens immediately following the region are parsed separately and removed from the collapsed cell-type name. Examples:

- `DG Granule` → region `DG`, layer `Unspecified`, cell type `Granule`
- `CA3 Pyramidal` → region `CA3`, layer `Unspecified`, cell type `Pyramidal`
- `MEC LII Stellate` → region `MEC`, layer `LII`, cell type `Stellate`
- `EC LI II Multipolar Pyramidal` → region `EC`, layer `LI-II`, cell type `Multipolar Pyramidal`
- `MEC LIII Multipolar Principal` → region `MEC`, layer `LIII`, cell type `Multipolar Principal`

The parser recognizes common Roman-numeral layer forms such as `LI`, `LII`, `LIII`, `LIV`, `LV`, `LVI`, as well as split forms such as `LI II` or `LII III`.

If your netlist already contains explicit region, layer, or cell-type columns, you can map them in `OPTIONAL_COL` instead of relying on this parser.


In [ ]:

def normalize_ei(x):
    """Normalize E/I labels to uppercase E or I."""
    s = str(x).strip().upper()
    aliases = {
        "EXCITATORY": "E",
        "EXC": "E",
        "+": "E",
        "INHIBITORY": "I",
        "INH": "I",
        "-": "I",
    }
    s = aliases.get(s, s)
    if s not in {"E", "I"}:
        raise ValueError(f"Unrecognized E/I label: {x!r}")
    return s


def infer_region(label):
    """Infer anatomical region from the first token of a Hippocampome-style label."""
    text = str(label).strip()
    if not text:
        return "UNKNOWN"
    return text.split()[0]


ROMAN_LAYER_TOKENS = {"I", "II", "III", "IV", "V", "VI"}
LAYER_PREFIX_RE = re.compile(r"^L(?P<roman>I{1,3}|IV|V|VI)$", flags=re.IGNORECASE)


def _normalize_layer_tokens(tokens):
    """
    Consume leading layer tokens after the region and return:
      (normalized_layer, number_of_consumed_tokens)

    Examples:
      ['LII', 'Stellate'] -> ('LII', 1)
      ['LI', 'II', 'Multipolar', 'Pyramidal'] -> ('LI-II', 2)
      ['LII', 'III', 'Pyramidal'] -> ('LII-LIII', 2)
    """
    if not tokens:
        return "Unspecified", 0

    first = tokens[0].upper()

    # Case 1: compact token such as LI, LII, LIII, LIV, LV, LVI.
    m = LAYER_PREFIX_RE.match(first)
    if m:
        layers = [f"L{m.group('roman').upper()}"]
        consumed = 1

        # Hippocampome labels sometimes continue with a bare Roman numeral:
        # "LI II" -> LI + LII.
        if len(tokens) > 1 and tokens[1].upper() in ROMAN_LAYER_TOKENS:
            layers.append(f"L{tokens[1].upper()}")
            consumed += 1

        return "-".join(layers), consumed

    # Case 2: a bare Roman numeral appears directly after region.
    if first in ROMAN_LAYER_TOKENS:
        layers = [f"L{first}"]
        consumed = 1
        if len(tokens) > 1 and tokens[1].upper() in ROMAN_LAYER_TOKENS:
            layers.append(f"L{tokens[1].upper()}")
            consumed += 1
        return "-".join(layers), consumed

    return "Unspecified", 0


def parse_region_layer_cell_type(label):
    """
    Parse a Hippocampome-style population label into region, layer, and
    region/layer-independent cell type.

    Examples
    --------
    EC LI II Multipolar Pyramidal -> EC, LI-LII, Multipolar Pyramidal
    MEC LIII Multipolar Principal -> MEC, LIII, Multipolar Principal
    CA3 Pyramidal -> CA3, Unspecified, Pyramidal
    """
    text = str(label).strip()
    if not text:
        return "UNKNOWN", "Unspecified", "UNKNOWN"

    parts = text.split()
    region = parts[0]
    remainder = parts[1:]

    layer, consumed = _normalize_layer_tokens(remainder)
    cell_tokens = remainder[consumed:]
    cell_type = " ".join(cell_tokens).strip()

    if not cell_type:
        cell_type = region if not remainder else "UNKNOWN"

    return region, layer, cell_type


def infer_layer(label):
    return parse_region_layer_cell_type(label)[1]


def infer_cell_type(label):
    """Remove region and leading layer notation from a Hippocampome label."""
    return parse_region_layer_cell_type(label)[2]


def _resolve_optional_metadata(df, side, neuron_col):
    region_col = OPTIONAL_COL[f"{side}_region"]
    layer_col = OPTIONAL_COL[f"{side}_layer"]
    cell_col = OPTIONAL_COL[f"{side}_cell_type"]

    if region_col is not None:
        if region_col not in df.columns:
            raise ValueError(f"Configured column {region_col!r} is not present.")
        region = df[region_col].astype(str).str.strip()
    else:
        region = df[neuron_col].map(infer_region)

    if layer_col is not None:
        if layer_col not in df.columns:
            raise ValueError(f"Configured column {layer_col!r} is not present.")
        layer = df[layer_col].astype(str).str.strip()
    else:
        layer = df[neuron_col].map(infer_layer)

    if cell_col is not None:
        if cell_col not in df.columns:
            raise ValueError(f"Configured column {cell_col!r} is not present.")
        cell_type = df[cell_col].astype(str).str.strip()
    else:
        cell_type = df[neuron_col].map(infer_cell_type)

    return region, layer, cell_type


def load_and_validate_netlist(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"{path} was not found. Put mij_netlist.csv beside the notebook, "
            "change DATA_PATH, or set the MIJ_NETLIST_PATH environment variable."
        )

    df = pd.read_csv(path).copy()

    required = set(COL.values())
    missing = required - set(df.columns)
    if missing:
        raise ValueError(
            f"Missing required columns: {sorted(missing)}\\n"
            f"Available columns: {list(df.columns)}"
        )

    # Standardize core fields.
    for key in ["pre", "post"]:
        df[COL[key]] = df[COL[key]].astype(str).str.strip()

    df[COL["pre_ei"]] = df[COL["pre_ei"]].map(normalize_ei)
    df[COL["post_ei"]] = df[COL["post_ei"]].map(normalize_ei)
    df[COL["weight"]] = pd.to_numeric(df[COL["weight"]], errors="coerce")

    if df[COL["weight"]].isna().any():
        bad = df[df[COL["weight"]].isna()]
        raise ValueError(
            f"{len(bad)} rows have nonnumeric or missing {COL['weight']!r} values."
        )

    if not np.isfinite(df[COL["weight"]]).all():
        raise ValueError("Weight column contains Inf/-Inf.")

    # Metadata.
    df["pre_region"], df["pre_layer"], df["pre_cell_type_collapsed"] = _resolve_optional_metadata(
        df, "pre", COL["pre"]
    )
    df["post_region"], df["post_layer"], df["post_cell_type_collapsed"] = _resolve_optional_metadata(
        df, "post", COL["post"]
    )

    return df


def build_node_metadata(df):
    """Create one validated metadata row per population."""
    pre = df[
        [COL["pre"], COL["pre_ei"], "pre_region", "pre_layer", "pre_cell_type_collapsed"]
    ].rename(
        columns={
            COL["pre"]: "node",
            COL["pre_ei"]: "ei",
            "pre_region": "region",
            "pre_layer": "layer",
            "pre_cell_type_collapsed": "cell_type",
        }
    )

    post = df[
        [COL["post"], COL["post_ei"], "post_region", "post_layer", "post_cell_type_collapsed"]
    ].rename(
        columns={
            COL["post"]: "node",
            COL["post_ei"]: "ei",
            "post_region": "region",
            "post_layer": "layer",
            "post_cell_type_collapsed": "cell_type",
        }
    )

    meta = pd.concat([pre, post], ignore_index=True).drop_duplicates()

    # Validate E/I identity.
    conflicts = meta.groupby("node")["ei"].nunique()
    bad = conflicts[conflicts > 1]
    if len(bad):
        raise ValueError(
            "Conflicting E/I assignments for: " + ", ".join(bad.index.astype(str))
        )

    # Validate region/cell-type identity too.
    for field in ["region", "layer", "cell_type"]:
        conflicts = meta.groupby("node")[field].nunique()
        bad = conflicts[conflicts > 1]
        if len(bad):
            warnings.warn(
                f"Conflicting {field} assignments for {len(bad)} nodes; "
                "the first observed value will be retained."
            )

    return (
        meta.sort_values("node")
        .drop_duplicates("node", keep="first")
        .reset_index(drop=True)
    )


def prepare_edges(df):
    """Collapse the netlist to one weighted directed edge per ordered population pair."""
    e = df[
        [
            COL["pre"], COL["post"], COL["pre_ei"], COL["post_ei"],
            COL["weight"], "pre_region", "post_region",
            "pre_layer", "post_layer",
            "pre_cell_type_collapsed", "post_cell_type_collapsed",
        ]
    ].copy()

    e = e.rename(
        columns={
            COL["pre"]: "source",
            COL["post"]: "target",
            COL["pre_ei"]: "source_ei",
            COL["post_ei"]: "target_ei",
            COL["weight"]: "weight",
            "pre_region": "source_region",
            "post_region": "target_region",
            "pre_layer": "source_layer",
            "post_layer": "target_layer",
            "pre_cell_type_collapsed": "source_cell_type",
            "post_cell_type_collapsed": "target_cell_type",
        }
    )

    if EXCLUDE_SELF_EDGES:
        e = e[e["source"] != e["target"]].copy()

    if AGGREGATE_DUPLICATE_EDGES:
        # Metadata should be identical for duplicated source/target rows.
        group_cols = [
            "source", "target", "source_ei", "target_ei",
            "source_region", "target_region",
            "source_layer", "target_layer",
            "source_cell_type", "target_cell_type",
        ]
        e = e.groupby(group_cols, as_index=False, dropna=False)["weight"].sum()

    e["edge_signature"] = e["source_ei"] + e["target_ei"]
    e["abs_weight"] = e["weight"].abs()
    return e.reset_index(drop=True)


def enumerate_3node_chains(edges, simple_only=True):
    """
    Enumerate every directed A->B->C chain by joining incoming and outgoing
    edges at the middle node B.
    """
    left = edges.rename(
        columns={
            "source": "node_1",
            "target": "node_2",
            "source_ei": "ei_1",
            "target_ei": "ei_2",
            "source_region": "region_1",
            "target_region": "region_2",
            "source_layer": "layer_1",
            "target_layer": "layer_2",
            "source_cell_type": "cell_type_1",
            "target_cell_type": "cell_type_2",
            "weight": "w_12",
            "abs_weight": "abs_w_12",
        }
    )[
        [
            "node_1", "node_2", "ei_1", "ei_2",
            "region_1", "region_2",
            "layer_1", "layer_2",
            "cell_type_1", "cell_type_2",
            "w_12", "abs_w_12",
        ]
    ]

    right = edges.rename(
        columns={
            "source": "node_2",
            "target": "node_3",
            "source_ei": "ei_2_right",
            "target_ei": "ei_3",
            "source_region": "region_2_right",
            "target_region": "region_3",
            "source_layer": "layer_2_right",
            "target_layer": "layer_3",
            "source_cell_type": "cell_type_2_right",
            "target_cell_type": "cell_type_3",
            "weight": "w_23",
            "abs_weight": "abs_w_23",
        }
    )[
        [
            "node_2", "node_3", "ei_2_right", "ei_3",
            "region_2_right", "region_3",
            "layer_2_right", "layer_3",
            "cell_type_2_right", "cell_type_3",
            "w_23", "abs_w_23",
        ]
    ]

    chains = left.merge(right, on="node_2", how="inner")

    # Internal consistency at the joined middle node.
    mismatch = chains["ei_2"] != chains["ei_2_right"]
    if mismatch.any():
        raise ValueError("E/I metadata mismatch at joined middle nodes.")

    if simple_only:
        chains = chains[
            (chains["node_1"] != chains["node_2"])
            & (chains["node_2"] != chains["node_3"])
            & (chains["node_1"] != chains["node_3"])
        ].copy()

    chains["signature"] = chains["ei_1"] + chains["ei_2"] + chains["ei_3"]

    chains["signed_chain_weight"] = chains["w_12"] * chains["w_23"]
    chains["abs_chain_weight"] = chains["abs_w_12"] * chains["abs_w_23"]
    chains["intensity_geomean"] = np.sqrt(chains["abs_chain_weight"])

    chains["region_pathway"] = (
        chains["region_1"].astype(str)
        + " → " + chains["region_2"].astype(str)
        + " → " + chains["region_3"].astype(str)
    )

    chains["layer_pathway"] = (
        chains["layer_1"].astype(str)
        + " → " + chains["layer_2"].astype(str)
        + " → " + chains["layer_3"].astype(str)
    )
    chains["region_layer_pathway"] = (
        chains["region_1"].astype(str) + ":" + chains["layer_1"].astype(str)
        + " → " + chains["region_2"].astype(str) + ":" + chains["layer_2"].astype(str)
        + " → " + chains["region_3"].astype(str) + ":" + chains["layer_3"].astype(str)
    )
    chains["cell_type_pathway"] = (
        chains["cell_type_1"].astype(str)
        + " → " + chains["cell_type_2"].astype(str)
        + " → " + chains["cell_type_3"].astype(str)
    )
    chains["node_pathway"] = (
        chains["node_1"].astype(str)
        + " → " + chains["node_2"].astype(str)
        + " → " + chains["node_3"].astype(str)
    )

    return chains.reset_index(drop=True)


def signature_summary(chains):
    if len(chains) == 0:
        raise ValueError("No valid chains were found.")

    total_count = len(chains)
    total_abs = chains["abs_chain_weight"].sum()

    s = (
        chains.groupby("signature", as_index=False)
        .agg(
            chain_count=("signature", "size"),
            abs_chain_weight=("abs_chain_weight", "sum"),
            signed_chain_weight=("signed_chain_weight", "sum"),
            mean_abs_chain_weight=("abs_chain_weight", "mean"),
            median_abs_chain_weight=("abs_chain_weight", "median"),
            mean_intensity=("intensity_geomean", "mean"),
            median_intensity=("intensity_geomean", "median"),
        )
        .set_index("signature")
        .reindex(SIGNATURE_ORDER, fill_value=0)
        .reset_index()
    )

    s["chain_count_pct"] = 100 * s["chain_count"] / total_count
    s["abs_chain_weight_pct"] = np.where(
        total_abs > 0, 100 * s["abs_chain_weight"] / total_abs, np.nan
    )
    s["signed_to_abs_ratio"] = np.where(
        s["abs_chain_weight"] > 0,
        s["signed_chain_weight"] / s["abs_chain_weight"],
        np.nan,
    )

    return s


def expand_node_participation(chains):
    """
    Expand each chain into 3 node-position rows.

    One third of each chain's weight is allocated to each node so that
    allocated weights sum back to the original total chain weight.
    """
    parts = []
    position_map = {1: "source", 2: "middle", 3: "target"}

    for k, pos in position_map.items():
        part = pd.DataFrame(
            {
                "signature": chains["signature"].to_numpy(),
                "position": pos,
                "node": chains[f"node_{k}"].to_numpy(),
                "ei": chains[f"ei_{k}"].to_numpy(),
                "region": chains[f"region_{k}"].to_numpy(),
                "layer": chains[f"layer_{k}"].to_numpy(),
                "cell_type": chains[f"cell_type_{k}"].to_numpy(),
                "abs_chain_weight_allocated": chains["abs_chain_weight"].to_numpy() / 3.0,
                "signed_chain_weight_allocated": chains["signed_chain_weight"].to_numpy() / 3.0,
            }
        )
        part["region_layer"] = part["region"].astype(str) + ":" + part["layer"].astype(str)
        parts.append(part)

    return pd.concat(parts, ignore_index=True)


def composition_summary(participation, group_col):
    """
    Summarize region or collapsed cell-type participation within each E/I signature.
    """
    g = (
        participation.groupby(["signature", group_col], as_index=False)
        .agg(
            node_occurrences=("node", "size"),
            unique_populations=("node", "nunique"),
            abs_chain_weight_allocated=("abs_chain_weight_allocated", "sum"),
            signed_chain_weight_allocated=("signed_chain_weight_allocated", "sum"),
        )
    )

    g["occurrence_pct_within_signature"] = (
        100
        * g["node_occurrences"]
        / g.groupby("signature")["node_occurrences"].transform("sum")
    )
    g["weight_pct_within_signature"] = (
        100
        * g["abs_chain_weight_allocated"]
        / g.groupby("signature")["abs_chain_weight_allocated"].transform("sum")
    )
    return g.sort_values(
        ["signature", "abs_chain_weight_allocated"], ascending=[True, False]
    ).reset_index(drop=True)


def overall_participation_summary(participation, group_col):
    g = (
        participation.groupby(group_col, as_index=False)
        .agg(
            node_occurrences=("node", "size"),
            unique_populations=("node", "nunique"),
            abs_chain_weight_allocated=("abs_chain_weight_allocated", "sum"),
            signed_chain_weight_allocated=("signed_chain_weight_allocated", "sum"),
        )
    )
    g["occurrence_pct"] = 100 * g["node_occurrences"] / g["node_occurrences"].sum()
    g["weight_pct"] = (
        100
        * g["abs_chain_weight_allocated"]
        / g["abs_chain_weight_allocated"].sum()
    )
    return g.sort_values("abs_chain_weight_allocated", ascending=False).reset_index(drop=True)


def pathway_summary(chains, pathway_col):
    g = (
        chains.groupby(pathway_col, as_index=False)
        .agg(
            chain_count=(pathway_col, "size"),
            abs_chain_weight=("abs_chain_weight", "sum"),
            signed_chain_weight=("signed_chain_weight", "sum"),
            mean_intensity=("intensity_geomean", "mean"),
        )
    )
    g["chain_count_pct"] = 100 * g["chain_count"] / g["chain_count"].sum()
    g["abs_chain_weight_pct"] = (
        100 * g["abs_chain_weight"] / g["abs_chain_weight"].sum()
    )
    return g.sort_values("abs_chain_weight", ascending=False).reset_index(drop=True)


def signature_pathway_summary(chains, pathway_col):
    g = (
        chains.groupby(["signature", pathway_col], as_index=False)
        .agg(
            chain_count=(pathway_col, "size"),
            abs_chain_weight=("abs_chain_weight", "sum"),
            signed_chain_weight=("signed_chain_weight", "sum"),
        )
    )
    g["weight_pct_within_signature"] = (
        100
        * g["abs_chain_weight"]
        / g.groupby("signature")["abs_chain_weight"].transform("sum")
    )
    g["count_pct_within_signature"] = (
        100
        * g["chain_count"]
        / g.groupby("signature")["chain_count"].transform("sum")
    )
    return g.sort_values(
        ["signature", "abs_chain_weight"], ascending=[True, False]
    ).reset_index(drop=True)


def savefig(name):
    if SAVE_FIGURES:
        plt.savefig(FIG_DIR / f"{name}.png", dpi=200, bbox_inches="tight")


def barh(df, label_col, value_col, title, xlabel, filename=None, top_n=None):
    plot_df = df.copy()
    if top_n is not None:
        plot_df = plot_df.nlargest(top_n, value_col)
    plot_df = plot_df.sort_values(value_col, ascending=True)

    fig, ax = plt.subplots(figsize=(10, max(4, 0.38 * len(plot_df))))
    ax.barh(plot_df[label_col].astype(str), plot_df[value_col])
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("")
    fig.tight_layout()
    if filename:
        savefig(filename)
    plt.show()


def heatmap_from_pivot(pivot, title, cbar_label, filename=None, figsize=None):
    if pivot.empty:
        print(f"No data available for: {title}")
        return

    if figsize is None:
        figsize = (max(8, 0.8 * len(pivot.columns)), max(4, 0.55 * len(pivot.index)))

    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(pivot.to_numpy(), aspect="auto")
    ax.set_xticks(np.arange(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=45, ha="right")
    ax.set_yticks(np.arange(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_title(title)
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label(cbar_label)
    fig.tight_layout()
    if filename:
        savefig(filename)
    plt.show()


def plot_signature_position_composition(participation, signature, group_col, top_n=12):
    subset = participation[participation["signature"] == signature].copy()
    if subset.empty:
        print(f"{signature}: no chains")
        return

    weighted = (
        subset.groupby(["position", group_col], as_index=False)["abs_chain_weight_allocated"]
        .sum()
    )

    # Convert to percent within each chain position.
    weighted["pct"] = (
        100
        * weighted["abs_chain_weight_allocated"]
        / weighted.groupby("position")["abs_chain_weight_allocated"].transform("sum")
    )

    top_groups = (
        weighted.groupby(group_col)["abs_chain_weight_allocated"]
        .sum()
        .nlargest(top_n)
        .index
    )
    weighted = weighted[weighted[group_col].isin(top_groups)]

    pivot = weighted.pivot(index="position", columns=group_col, values="pct").fillna(0)
    pivot = pivot.reindex(["source", "middle", "target"]).fillna(0)

    fig, ax = plt.subplots(figsize=(11, 5))
    pivot.plot(kind="bar", stacked=True, ax=ax)
    ax.set_title(f"{signature}: {group_col.replace('_', ' ')} composition by chain position")
    ax.set_xlabel("Chain position")
    ax.set_ylabel("Absolute chain-weight share within position (%)")
    ax.legend(title=group_col.replace("_", " "), bbox_to_anchor=(1.02, 1), loc="upper left")
    fig.tight_layout()
    savefig(f"{signature}_{group_col}_by_position")
    plt.show()


## 3. Load, validate, and profile the netlist

In [ ]:

netlist = load_and_validate_netlist(DATA_PATH)
nodes = build_node_metadata(netlist)
edges = prepare_edges(netlist)

print(f"Netlist rows:             {len(netlist):,}")
print(f"Unique populations:       {nodes['node'].nunique():,}")
print(f"Directed non-self edges:  {len(edges):,}")
print(f"Excitatory populations:   {(nodes['ei'] == 'E').sum():,}")
print(f"Inhibitory populations:   {(nodes['ei'] == 'I').sum():,}")
print(f"Regions:                  {nodes['region'].nunique():,}")
print(f"Layers:                   {nodes['layer'].nunique():,}")
print(f"Collapsed cell types:     {nodes['cell_type'].nunique():,}")

display(nodes.head(15))
display(edges.head(15))


In [ ]:

# Data-quality audit: does the numerical edge sign agree with presynaptic E/I identity?
nonzero = edges[edges["weight"] != 0].copy()
nonzero["expected_sign"] = np.where(nonzero["source_ei"] == "E", 1, -1)
nonzero["observed_sign"] = np.sign(nonzero["weight"]).astype(int)
nonzero["sign_concordant"] = nonzero["expected_sign"] == nonzero["observed_sign"]

sign_audit = (
    nonzero.groupby("source_ei", as_index=False)
    .agg(
        nonzero_edges=("weight", "size"),
        concordant_edges=("sign_concordant", "sum"),
    )
)
sign_audit["concordance_pct"] = (
    100 * sign_audit["concordant_edges"] / sign_audit["nonzero_edges"]
)

display(sign_audit)

mismatches = nonzero[~nonzero["sign_concordant"]]
print(f"Numerical-sign / presynaptic-EI mismatches: {len(mismatches):,}")
if len(mismatches):
    display(mismatches.head(20))


In [ ]:

# Baseline edge-class summary before chain enumeration.
edge_class_summary = (
    edges.groupby("edge_signature", as_index=False)
    .agg(
        edge_count=("edge_signature", "size"),
        abs_edge_weight=("abs_weight", "sum"),
        signed_edge_weight=("weight", "sum"),
    )
)
edge_class_summary["edge_count_pct"] = (
    100 * edge_class_summary["edge_count"] / edge_class_summary["edge_count"].sum()
)
edge_class_summary["abs_edge_weight_pct"] = (
    100 * edge_class_summary["abs_edge_weight"] / edge_class_summary["abs_edge_weight"].sum()
)

display(edge_class_summary.sort_values("edge_signature"))

barh(
    edge_class_summary,
    "edge_signature",
    "edge_count",
    "Directed E/I edge counts",
    "Edge count",
    filename="00_edge_class_counts",
)
barh(
    edge_class_summary,
    "edge_signature",
    "abs_edge_weight_pct",
    "Absolute edge-weight share by E/I edge class",
    "Absolute edge-weight share (%)",
    filename="01_edge_class_weight_share",
)


## 4. Enumerate all 3-node directed chains

In [ ]:

chains = enumerate_3node_chains(edges, simple_only=SIMPLE_CHAINS_ONLY)

print(f"Enumerated chains: {len(chains):,}")
print(f"Unique node pathways: {chains['node_pathway'].nunique():,}")
print(f"Unique region pathways: {chains['region_pathway'].nunique():,}")
print(f"Unique collapsed-cell-type pathways: {chains['cell_type_pathway'].nunique():,}")

display(
    chains[
        [
            "node_pathway", "signature", "w_12", "w_23",
            "signed_chain_weight", "abs_chain_weight",
            "intensity_geomean", "region_pathway", "layer_pathway",
            "region_layer_pathway", "cell_type_pathway"
        ]
    ].head(20)
)


## 5. E/I signature statistics

In [ ]:

sig = signature_summary(chains)
display(sig)

sig.to_csv(OUTPUT_DIR / "ei_signature_summary.csv", index=False)


In [ ]:

barh(
    sig,
    "signature",
    "chain_count",
    "Count of directed 3-node chains by E/I signature",
    "Chain count",
    filename="02_signature_chain_count",
)

barh(
    sig,
    "signature",
    "chain_count_pct",
    "Share of chain count by E/I signature",
    "Chain count (%)",
    filename="03_signature_chain_count_pct",
)

barh(
    sig,
    "signature",
    "abs_chain_weight_pct",
    "Share of total absolute chain weight by E/I signature",
    "Absolute chain-weight share (%)",
    filename="04_signature_abs_weight_pct",
)

barh(
    sig,
    "signature",
    "signed_chain_weight",
    "Net signed chain weight by E/I signature",
    "Signed chain-weight sum",
    filename="05_signature_signed_weight",
)

barh(
    sig,
    "signature",
    "mean_intensity",
    "Mean weighted-chain intensity by E/I signature",
    "Mean geometric-mean intensity",
    filename="06_signature_mean_intensity",
)



### How to read the signature summary

- `chain_count_pct`: structural prevalence of the signature.
- `abs_chain_weight_pct`: how much of total two-edge coupling strength is carried by that signature.
- `signed_chain_weight`: direction of the net multiplicative effect after preserving signs.
- `signed_to_abs_ratio`: ranges from approximately `-1` to `+1`; values near zero indicate substantial cancellation among signed chain products.
- `mean_intensity`: typical weighted strength of chains in that signature, independent of how many chains exist.

A signature can therefore be **rare but strong**, **common but weak**, or both common and strong.


## 6. Region composition

In [ ]:

participation = expand_node_participation(chains)

sig_region = composition_summary(participation, "region")
overall_region = overall_participation_summary(participation, "region")
region_paths = pathway_summary(chains, "region_pathway")
sig_region_paths = signature_pathway_summary(chains, "region_pathway")

display(overall_region)
display(sig_region.head(50))
display(region_paths.head(TOP_N_PATHWAYS))

overall_region.to_csv(OUTPUT_DIR / "region_participation_overall.csv", index=False)
sig_region.to_csv(OUTPUT_DIR / "region_composition_by_signature.csv", index=False)
region_paths.to_csv(OUTPUT_DIR / "region_pathway_summary.csv", index=False)
sig_region_paths.to_csv(OUTPUT_DIR / "region_pathways_by_signature.csv", index=False)


In [ ]:

barh(
    overall_region,
    "region",
    "weight_pct",
    "Overall regional makeup of chain weight",
    "Allocated absolute chain-weight share (%)",
    filename="07_overall_region_weight_share",
)

barh(
    overall_region,
    "region",
    "occurrence_pct",
    "Overall regional makeup of chain-node occurrences",
    "Node-occurrence share (%)",
    filename="08_overall_region_occurrence_share",
)

barh(
    region_paths,
    "region_pathway",
    "abs_chain_weight_pct",
    f"Top {TOP_N_PATHWAYS} region pathways by absolute chain weight",
    "Absolute chain-weight share (%)",
    filename="09_top_region_pathways",
    top_n=TOP_N_PATHWAYS,
)


In [ ]:

# Heatmap: within each E/I signature, what percentage of allocated chain weight
# is associated with each region?
region_heat = (
    sig_region.pivot(
        index="signature",
        columns="region",
        values="weight_pct_within_signature",
    )
    .reindex(SIGNATURE_ORDER)
    .fillna(0)
)

heatmap_from_pivot(
    region_heat,
    "Regional composition within each E/I chain signature",
    "Within-signature chain-weight share (%)",
    filename="10_signature_region_heatmap",
)


## 7. Cell-type composition regardless of region

In [ ]:

sig_cell = composition_summary(participation, "cell_type")
overall_cell = overall_participation_summary(participation, "cell_type")
cell_paths = pathway_summary(chains, "cell_type_pathway")
sig_cell_paths = signature_pathway_summary(chains, "cell_type_pathway")

display(overall_cell.head(50))
display(sig_cell.head(100))
display(cell_paths.head(TOP_N_PATHWAYS))

overall_cell.to_csv(OUTPUT_DIR / "cell_type_participation_overall.csv", index=False)
sig_cell.to_csv(OUTPUT_DIR / "cell_type_composition_by_signature.csv", index=False)
cell_paths.to_csv(OUTPUT_DIR / "cell_type_pathway_summary.csv", index=False)
sig_cell_paths.to_csv(OUTPUT_DIR / "cell_type_pathways_by_signature.csv", index=False)


In [ ]:

barh(
    overall_cell,
    "cell_type",
    "weight_pct",
    f"Top {TOP_N_CELL_TYPES} cell types by chain-weight participation",
    "Allocated absolute chain-weight share (%)",
    filename="11_top_cell_types_weight_share",
    top_n=TOP_N_CELL_TYPES,
)

barh(
    overall_cell,
    "cell_type",
    "occurrence_pct",
    f"Top {TOP_N_CELL_TYPES} cell types by chain-node occurrence",
    "Node-occurrence share (%)",
    filename="12_top_cell_types_occurrence_share",
    top_n=TOP_N_CELL_TYPES,
)

barh(
    cell_paths,
    "cell_type_pathway",
    "abs_chain_weight_pct",
    f"Top {TOP_N_PATHWAYS} collapsed cell-type pathways",
    "Absolute chain-weight share (%)",
    filename="13_top_cell_type_pathways",
    top_n=TOP_N_PATHWAYS,
)


In [ ]:

# Choose the globally strongest cell types, then show their contribution
# within every E/I signature.
top_cell_types = overall_cell.head(TOP_N_CELL_TYPES)["cell_type"]

cell_heat = (
    sig_cell[sig_cell["cell_type"].isin(top_cell_types)]
    .pivot(
        index="signature",
        columns="cell_type",
        values="weight_pct_within_signature",
    )
    .reindex(SIGNATURE_ORDER)
    .fillna(0)
)

heatmap_from_pivot(
    cell_heat,
    f"Top {TOP_N_CELL_TYPES} cell types within each E/I signature",
    "Within-signature chain-weight share (%)",
    filename="14_signature_cell_type_heatmap",
    figsize=(max(12, 0.7 * len(cell_heat.columns)), 6),
)


## 8. Region × layer analysis

This section treats the anatomical **region** and cortical/hippocampal **layer** as separate attributes.

Examples:

- `EC LI II Multipolar Pyramidal` contributes to region `EC`, layer `LI-LII`, cell type `Multipolar Pyramidal`.
- `MEC LIII Multipolar Principal` contributes to region `MEC`, layer `LIII`, cell type `Multipolar Principal`.

This makes it possible to ask whether a chain signature is concentrated not merely in a region, but in a specific **region-layer compartment**.


In [ ]:

# Overall layer and region-layer participation.
sig_layer = composition_summary(participation, "layer")
overall_layer = overall_participation_summary(participation, "layer")

sig_region_layer = composition_summary(participation, "region_layer")
overall_region_layer = overall_participation_summary(participation, "region_layer")

layer_paths = pathway_summary(chains, "layer_pathway")
region_layer_paths = pathway_summary(chains, "region_layer_pathway")
sig_region_layer_paths = signature_pathway_summary(chains, "region_layer_pathway")

display(overall_layer)
display(overall_region_layer.head(50))
display(region_layer_paths.head(TOP_N_PATHWAYS))

overall_layer.to_csv(OUTPUT_DIR / "layer_participation_overall.csv", index=False)
sig_layer.to_csv(OUTPUT_DIR / "layer_composition_by_signature.csv", index=False)
overall_region_layer.to_csv(OUTPUT_DIR / "region_layer_participation_overall.csv", index=False)
sig_region_layer.to_csv(OUTPUT_DIR / "region_layer_composition_by_signature.csv", index=False)
layer_paths.to_csv(OUTPUT_DIR / "layer_pathway_summary.csv", index=False)
region_layer_paths.to_csv(OUTPUT_DIR / "region_layer_pathway_summary.csv", index=False)
sig_region_layer_paths.to_csv(
    OUTPUT_DIR / "region_layer_pathways_by_signature.csv", index=False
)


In [ ]:

barh(
    overall_layer,
    "layer",
    "weight_pct",
    "Overall layer makeup of chain weight",
    "Allocated absolute chain-weight share (%)",
    filename="17_overall_layer_weight_share",
)

barh(
    overall_region_layer,
    "region_layer",
    "weight_pct",
    f"Top {TOP_N_PATHWAYS} region-layer compartments by chain weight",
    "Allocated absolute chain-weight share (%)",
    filename="18_top_region_layer_weight_share",
    top_n=TOP_N_PATHWAYS,
)

barh(
    region_layer_paths,
    "region_layer_pathway",
    "abs_chain_weight_pct",
    f"Top {TOP_N_PATHWAYS} region-layer pathways",
    "Absolute chain-weight share (%)",
    filename="19_top_region_layer_pathways",
    top_n=TOP_N_PATHWAYS,
)


In [ ]:

# Heatmap: layer composition within each E/I signature.
layer_heat = (
    sig_layer.pivot(
        index="signature",
        columns="layer",
        values="weight_pct_within_signature",
    )
    .reindex(SIGNATURE_ORDER)
    .fillna(0)
)

heatmap_from_pivot(
    layer_heat,
    "Layer composition within each E/I chain signature",
    "Within-signature chain-weight share (%)",
    filename="20_signature_layer_heatmap",
)


In [ ]:

# Heatmap: strongest region-layer compartments within each E/I signature.
top_region_layers = overall_region_layer.head(TOP_N_CELL_TYPES)["region_layer"]

region_layer_heat = (
    sig_region_layer[sig_region_layer["region_layer"].isin(top_region_layers)]
    .pivot(
        index="signature",
        columns="region_layer",
        values="weight_pct_within_signature",
    )
    .reindex(SIGNATURE_ORDER)
    .fillna(0)
)

heatmap_from_pivot(
    region_layer_heat,
    f"Top {TOP_N_CELL_TYPES} region-layer compartments within each E/I signature",
    "Within-signature chain-weight share (%)",
    filename="21_signature_region_layer_heatmap",
    figsize=(max(12, 0.75 * len(region_layer_heat.columns)), 6),
)


In [ ]:

# Region x layer matrix across all chains.
region_layer_matrix = (
    participation.groupby(["region", "layer"], as_index=False)
    ["abs_chain_weight_allocated"].sum()
)

region_layer_pivot = (
    region_layer_matrix.pivot(
        index="region",
        columns="layer",
        values="abs_chain_weight_allocated",
    )
    .fillna(0)
)

# Convert each region to percentages across its own layers.
region_layer_pct = region_layer_pivot.div(
    region_layer_pivot.sum(axis=1).replace(0, np.nan),
    axis=0
) * 100
region_layer_pct = region_layer_pct.fillna(0)

heatmap_from_pivot(
    region_layer_pct,
    "Layer composition within each anatomical region",
    "Within-region chain-weight share (%)",
    filename="22_region_by_layer_heatmap",
)


In [ ]:

# Position-specific region-layer composition for every observed signature.
if PLOT_EACH_SIGNATURE:
    observed = set(chains["signature"])
    for signature in SIGNATURE_ORDER:
        if signature not in observed:
            continue

        plot_signature_position_composition(
            participation,
            signature,
            "layer",
            top_n=12,
        )
        plot_signature_position_composition(
            participation,
            signature,
            "region_layer",
            top_n=12,
        )


### Interpreting region × layer results

Use these views to separate three different questions:

1. **Region effect:** Is a signature concentrated in DG, CA3, CA1, EC, MEC, SUB, etc.?
2. **Layer effect:** Is it concentrated in superficial or deeper layers independent of region?
3. **Region-layer interaction:** Is a signature specifically concentrated in compartments such as `EC:LI-LII` or `MEC:LIII`?

This is especially useful when the same broad cell class occurs in multiple regions or layers. The collapsed cell-type analysis now removes both region and leading layer notation, while this section preserves layer information explicitly rather than discarding it.


## 9. Region-to-region chain flow

In [ ]:

# Collapse each 3-node chain to its source and target regions.
# The middle region is ignored in this particular summary.
region_flow = (
    chains.groupby(["region_1", "region_3"], as_index=False)
    .agg(
        chain_count=("signature", "size"),
        abs_chain_weight=("abs_chain_weight", "sum"),
    )
)

flow_pivot = (
    region_flow.pivot(index="region_1", columns="region_3", values="abs_chain_weight")
    .fillna(0)
)

heatmap_from_pivot(
    flow_pivot,
    "Two-edge chain weight from source region to target region",
    "Absolute chain weight",
    filename="15_region_source_target_flow",
)


## 10. Position-specific composition for every E/I signature

In [ ]:

# These plots distinguish source, middle, and target composition.
# Example: for EII, they show which regions/cell types occupy the E source,
# first I middle node, and second I target node.

if PLOT_EACH_SIGNATURE:
    observed = set(chains["signature"])
    for signature in SIGNATURE_ORDER:
        if signature not in observed:
            print(f"{signature}: no observed chains")
            continue

        plot_signature_position_composition(
            participation,
            signature,
            "region",
            top_n=12,
        )
        plot_signature_position_composition(
            participation,
            signature,
            "cell_type",
            top_n=12,
        )


## 11. Drill-down tables for a selected signature

In [ ]:

SELECT_SIGNATURE = "EII"

if SELECT_SIGNATURE not in SIGNATURE_ORDER:
    raise ValueError(f"SELECT_SIGNATURE must be one of {SIGNATURE_ORDER}")

selected = chains[chains["signature"] == SELECT_SIGNATURE].copy()

print(f"{SELECT_SIGNATURE} chains: {len(selected):,}")
print(
    f"{SELECT_SIGNATURE} share of all absolute chain weight: "
    f"{sig.set_index('signature').loc[SELECT_SIGNATURE, 'abs_chain_weight_pct']:.2f}%"
)

print("\nTop individual chains:")
display(
    selected.nlargest(25, "abs_chain_weight")[
        [
            "node_pathway", "region_pathway", "cell_type_pathway",
            "w_12", "w_23", "signed_chain_weight",
            "abs_chain_weight", "intensity_geomean"
        ]
    ]
)

print("\nTop regions within signature:")
display(
    sig_region[sig_region["signature"] == SELECT_SIGNATURE]
    .head(25)
)

print("\nTop collapsed cell types within signature:")
display(
    sig_cell[sig_cell["signature"] == SELECT_SIGNATURE]
    .head(25)
)

print("\nTop region pathways within signature:")
display(
    sig_region_paths[sig_region_paths["signature"] == SELECT_SIGNATURE]
    .head(25)
)

print("\nTop cell-type pathways within signature:")
display(
    sig_cell_paths[sig_cell_paths["signature"] == SELECT_SIGNATURE]
    .head(25)
)


## 12. Compare count share with weight share

In [ ]:

# Positive values indicate a signature carries more chain weight than expected
# from its prevalence alone; negative values indicate the reverse.
sig_compare = sig.copy()
sig_compare["weight_minus_count_pct_points"] = (
    sig_compare["abs_chain_weight_pct"] - sig_compare["chain_count_pct"]
)
sig_compare = sig_compare.sort_values("weight_minus_count_pct_points", ascending=False)

display(
    sig_compare[
        [
            "signature",
            "chain_count_pct",
            "abs_chain_weight_pct",
            "weight_minus_count_pct_points",
            "mean_intensity",
        ]
    ]
)

barh(
    sig_compare,
    "signature",
    "weight_minus_count_pct_points",
    "Chain-weight share minus chain-count share",
    "Percentage-point difference",
    filename="16_signature_weight_minus_count",
)


## 13. Export all analysis tables

In [ ]:

edge_class_summary.to_csv(OUTPUT_DIR / "edge_class_summary.csv", index=False)
nodes.to_csv(OUTPUT_DIR / "node_metadata.csv", index=False)
sig_compare.to_csv(OUTPUT_DIR / "ei_signature_count_vs_weight.csv", index=False)
region_flow.to_csv(OUTPUT_DIR / "region_source_target_flow.csv", index=False)

if EXPORT_CHAIN_TABLE:
    chains.to_csv(OUTPUT_DIR / "all_3node_chains.csv", index=False)
    participation.to_csv(OUTPUT_DIR / "all_chain_node_participation.csv", index=False)

print("Exported summary files:")
for p in sorted(OUTPUT_DIR.glob("*.csv")):
    print(" -", p.name)

if SAVE_FIGURES:
    print(f"\nSaved figures: {len(list(FIG_DIR.glob('*.png')))}")
    print("Figure directory:", FIG_DIR.resolve())



## 13. Suggested interpretation sequence

1. **Start with `ei_signature_summary.csv`.**  
   Identify signatures that are common by count versus dominant by weight.

2. **Compare count share and weight share.**  
   A large positive `weight_minus_count_pct_points` indicates a signature is disproportionately strong.

3. **Inspect the region heatmap.**  
   This shows whether a signature is concentrated in DG, CA3, CA1, EC/MEC, SUB, or another anatomical compartment.

4. **Inspect the collapsed cell-type heatmap.**  
   This determines whether signatures are driven by recurring cell classes regardless of anatomical region.

5. **Use position-specific plots.**  
   These distinguish whether a region or cell type appears mainly as the chain source, intermediary, or target.

6. **Inspect top region and cell-type pathways.**  
   This moves from composition to actual ordered pathways.

7. **Drill into individual high-weight chains.**  
   Set `SELECT_SIGNATURE` to signatures such as `EII`, `IEI`, or `EEI` and inspect the strongest node-level chains.

### Important interpretation note

This is a **chain/path analysis**, not an induced-triad census. A chain `A → B → C` is counted whenever those two ordered edges exist, even if other edges among A, B, and C also exist. That makes this analysis appropriate for questions about sequential coupling and E/I routing. If the question changes to *which complete 3-node directed motif is present*, use the separate directed-triad / Rees-style motif census workflow.
